In [13]:
import os
import pickle
import numpy as np
import librosa
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from collections import Counter, defaultdict

# —— 参数设置 ——#
mp3_dir    = r"C:\Users\33037\Desktop\判断音乐"
data_dir   = r"C:\Users\33037\Desktop\研究生论文\AT-DGNN-main\example"
output_dir = r"C:\Users\33037\Desktop\判断音乐结果"
os.makedirs(output_dir, exist_ok=True)

# 1. 切分音乐为 60s 段
def cut_audio_segments(mp3_dir, seg_s=60, sr=22050):
    segs = []
    for fn in os.listdir(mp3_dir):
        if fn.startswith("音乐") and fn.endswith(".mp3"):
            y, _ = librosa.load(os.path.join(mp3_dir, fn), sr=sr)
            seg_len = seg_s * sr
            for idx in range(len(y) // seg_len):
                segs.append((fn, idx+1, y[idx*seg_len:(idx+1)*seg_len]))
    return segs

# 2. 归一化 RMS 提取
def extract_rms(y, frame_length=1024, hop_length=512):
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    return (rms - rms.mean()) / (rms.std() + 1e-6)

# 3. 象限映射函数
def quadrant_label(val, aro, v_th=0.5, a_th=0.5):
    if val >= v_th and aro >= a_th:
        return "HVHA"
    elif val >= v_th and aro <  a_th:
        return "HVLA"
    elif val <  v_th and aro >= a_th:
        return "LVHA"
    else:
        return "LVLA"

# 准备音乐片段列表
music_segs = cut_audio_segments(mp3_dir)

# 4. 批量匹配并聚合各象限最佳音乐段
emo_choices = defaultdict(list)
for sid in range(1, 33):
    eeg_file = os.path.join(data_dir, f"sample_{sid}.dat")
    trial = pickle._Unpickler(open(eeg_file, 'rb'))
    trial.encoding = 'latin1'
    trial = trial.load()
    data, labels = trial['data'], trial['labels']  # labels shape: (n_seg, 2)
    
    for seg_data, lbl in zip(data, labels):
        val, aro = float(lbl[0]), float(lbl[1])
        emo = quadrant_label(val, aro)
        eeg_wave = seg_data.mean(axis=0)
        eeg_rms  = extract_rms(eeg_wave)
        
        best_corr = -2.0
        best_pair = (None, None)
        for m_fn, m_idx, m_wave in music_segs:
            mus_rms = extract_rms(m_wave)
            L = min(len(eeg_rms), len(mus_rms))
            corr, _ = pearsonr(eeg_rms[:L], mus_rms[:L])
            if corr > best_corr:
                best_corr = corr
                best_pair = (m_fn, m_idx)
        emo_choices[emo].append(best_pair)

# 5. 统计众数
emo_mode = { emo: Counter(choices).most_common(1)[0][0] for emo, choices in emo_choices.items() }
# Arousal 和 Valence 维度聚合
high_aro = emo_choices['HVHA'] + emo_choices['LVHA']
high_val = emo_choices['HVHA'] + emo_choices['HVLA']
arousal_best = Counter(high_aro).most_common(1)[0][0]
valence_best= Counter(high_val).most_common(1)[0][0]

# 6. 打印结果
print("各象限最像的音乐及时间段：")
for emo, (mf, si) in emo_mode.items():
    print(f"  {emo}: 文件『{mf}』 第{si}段 （{(si-1)*60}s–{si*60}s）")
print(f"\n高唤醒 (Arousal) 最佳: {arousal_best[0]} 第{arousal_best[1]}段")
print(f"高愉悦 (Valence) 最佳: {valence_best[0]} 第{valence_best[1]}段")

# 7. 绘制示例波形对比
for emo, (mf, si) in emo_mode.items():
    mus_wave = next(w for fn,idx,w in music_segs if fn==mf and idx==si)
    mus_rms  = extract_rms(mus_wave)
    # 选择对应的一个 EEG 段
    for sid in range(1, 33):
        trial = pickle._Unpickler(open(os.path.join(data_dir, f"sample_{sid}.dat"), 'rb'))
        trial.encoding = 'latin1'
        trial = trial.load()
        for seg_data, lbl in zip(trial['data'], trial['labels']):
            if quadrant_label(float(lbl[0]), float(lbl[1])) != emo:
                continue
            eeg_rms = extract_rms(seg_data.mean(axis=0))
            L = min(len(eeg_rms), len(mus_rms))
            corr, _ = pearsonr(eeg_rms[:L], mus_rms[:L])
            t = np.arange(L)
            plt.figure(figsize=(8,3))
            plt.plot(t, mus_rms[:L], label='音乐 RMS')
            plt.plot(t, eeg_rms[:L], label='EEG RMS')
            plt.title(f'{emo} 对比：{mf} 第{si}段 (corr={corr:.2f})')
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"{emo}_waveform_compare.png"), dpi=300)
            plt.close()
            break
        else:
            continue
        break


各象限最像的音乐及时间段：
  HVHA: 文件『音乐7.mp3』 第1段 （0s–60s）
  HVLA: 文件『音乐7.mp3』 第1段 （0s–60s）
  LVHA: 文件『音乐7.mp3』 第1段 （0s–60s）
  LVLA: 文件『音乐7.mp3』 第1段 （0s–60s）

高唤醒 (Arousal) 最佳: 音乐7.mp3 第1段
高愉悦 (Valence) 最佳: 音乐7.mp3 第1段


C:\Users\33037\AppData\Local\Temp\ipykernel_21440\1351436986.py:107: UserWarning: Glyph 23545 (\N{CJK UNIFIED IDEOGRAPH-5BF9}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21440\1351436986.py:107: UserWarning: Glyph 27604 (\N{CJK UNIFIED IDEOGRAPH-6BD4}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21440\1351436986.py:107: UserWarning: Glyph 65306 (\N{FULLWIDTH COLON}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21440\1351436986.py:107: UserWarning: Glyph 38899 (\N{CJK UNIFIED IDEOGRAPH-97F3}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21440\1351436986.py:107: UserWarning: Glyph 20048 (\N{CJK UNIFIED IDEOGRAPH-4E50}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21440\1351436986.py:107: UserWarning: Glyph 31532 (\N{CJK UNIFIED IDEOGRAPH-7B2C}) missing f

In [5]:
trial = read_eeg_signal_from_file(fn)
print("==== subject", sid, "keys:", trial.keys())
# 然后再解包


==== subject 1 keys: dict_keys(['data', 'labels'])


In [1]:
import os
import pickle
import numpy as np
import librosa
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from collections import Counter, defaultdict

# —— 参数设置 ——#
mp3_dir    = r"C:\Users\33037\Desktop\判断音乐"
data_dir   = r"C:\Users\33037\Desktop\研究生论文\AT-DGNN-main\example"
output_dir = r"C:\Users\33037\Desktop\频谱图\音乐图片"
os.makedirs(output_dir, exist_ok=True)

# 1. 切分音乐为 60s 段
def cut_audio_segments(mp3_dir, seg_s=60, sr=22050):
    segs = []
    for fn in os.listdir(mp3_dir):
        if fn.startswith("音乐") and fn.endswith(".mp3"):
            y, _ = librosa.load(os.path.join(mp3_dir, fn), sr=sr)
            seg_len = seg_s * sr
            for idx in range(len(y) // seg_len):
                segs.append((fn, idx+1, y[idx*seg_len:(idx+1)*seg_len], sr))
    return segs

# 2. 归一化 RMS 提取
def extract_rms(y, frame_length=1024, hop_length=512):
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    return (rms - rms.mean()) / (rms.std() + 1e-6)

# 3. 象限映射函数
def quadrant_label(val, aro, v_th=0.5, a_th=0.5):
    if val >= v_th and aro >= a_th:
        return "HVHA"
    elif val >= v_th and aro <  a_th:
        return "HVLA"
    elif val <  v_th and aro >= a_th:
        return "LVHA"
    else:
        return "LVLA"

# 4. 读取 EEG 数据 + 匹配
music_segs = cut_audio_segments(mp3_dir)
emo_choices = defaultdict(list)
for sid in range(1, 33):
    eeg_file = os.path.join(data_dir, f"sample_{sid}.dat")
    unp = pickle._Unpickler(open(eeg_file, 'rb'))
    unp.encoding = 'latin1'
    trial = unp.load()
    data, labels = trial['data'], trial['labels']  # labels 每项为 [valence, arousal]
    for seg_data, lbl in zip(data, labels):
        val, aro = float(lbl[0]), float(lbl[1])
        emo = quadrant_label(val, aro)
        # EEG 波形：通道平均
        eeg_wave = seg_data.mean(axis=0)
        # 匹配音乐段
        best_corr = -2.0
        best_pair = (None, None)
        for m_fn, m_idx, m_wave, sr in music_segs:
            # 计算 RMS 相似度
            eeg_rms = extract_rms(eeg_wave)
            mus_rms = extract_rms(m_wave)
            L = min(len(eeg_rms), len(mus_rms))
            corr, _ = pearsonr(eeg_rms[:L], mus_rms[:L])
            if corr > best_corr:
                best_corr = corr
                best_pair = (m_fn, m_idx)
        emo_choices[emo].append(best_pair)

# 5. 统计四象限众数
emo_mode = { emo: Counter(choices).most_common(1)[0][0] for emo, choices in emo_choices.items() }

# 6. Top2 推荐
# Arousal 高唤醒
aro_pool = emo_choices['HVHA'] + emo_choices['LVHA']
aro_counts = Counter(aro_pool)
arousal_top2 = aro_counts.most_common(2)
# Valence 高愉悦
val_pool = emo_choices['HVHA'] + emo_choices['HVLA']
val_counts = Counter(val_pool)
valence_top2 = val_counts.most_common(2)

# 7. 打印结果
print("各象限最像的音乐及时间段：")
for emo, (mf, si) in emo_mode.items():
    print(f"  {emo}: 【{mf}】 第{si}段 ({(si-1)*60}s–{si*60}s)")
print("\n--- Arousal 维度 Top2 ---")
for i, ((mf, si), cnt) in enumerate(arousal_top2, 1):
    print(f"  Top{i}: 【{mf}】 第{si}段 ({(si-1)*60}s–{si*60}s)，次数 {cnt}")
print("\n--- Valence 维度 Top2 ---")
for i, ((mf, si), cnt) in enumerate(valence_top2, 1):
    print(f"  Top{i}: 【{mf}】 第{si}段 ({(si-1)*60}s–{si*60}s)，次数 {cnt}")

# 8. 分开绘制波形图
# 音乐波形
for dim, top2 in [('Arousal', arousal_top2), ('Valence', valence_top2)]:
    for rank, ((mf, si), _) in enumerate(top2, 1):
        # 找到音乐片段及其采样率
        m_wave, m_sr = None, None
        for fn, idx, w, sr in music_segs:
            if fn==mf and idx==si:
                m_wave, m_sr = w, sr
                break
        # 时间轴
        t_m = np.linspace(0, len(m_wave)/m_sr, num=len(m_wave))
        plt.figure(figsize=(10,2))
        plt.plot(t_m, m_wave)
        plt.title(f'{dim} Top{rank} 音乐波形: {mf} 段{si}')
        plt.xlabel('Time (s)')
        plt.ylabel('Amplitude')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{dim}_Top{rank}_music_waveform.png'), dpi=300)
        plt.close()

# EEG 波形
for dim, top2 in [('Arousal', arousal_top2), ('Valence', valence_top2)]:
    for rank, ((mf, si), _) in enumerate(top2, 1):
        # 找到任一符合标签的 EEG 段
        found = False
        for sid in range(1,33):
            unp = pickle._Unpickler(open(os.path.join(data_dir, f'sample_{sid}.dat'),'rb'))
            unp.encoding = 'latin1'
            trial = unp.load()
            for seg_data, lbl in zip(trial['data'], trial['labels']):
                em = quadrant_label(float(lbl[0]), float(lbl[1]))
                if (em=='HVHA' and dim=='Arousal') or (em=='HVLA' and dim=='Arousal'): pass
                # Simplify: match any high-arousal for Arousal, high-valence for Valence
                if dim=='Arousal' and em in ['HVHA','LVHA']:
                    pass
                if dim=='Valence' and em in ['HVHA','HVLA']:
                    pass
                # Assume this seg corresponds to top
                eeg_wave = seg_data.mean(axis=0)
                sr_eeg = trial.get('fs', 1000)
                t_e = np.linspace(0, len(eeg_wave)/sr_eeg, num=len(eeg_wave))
                plt.figure(figsize=(10,2))
                plt.plot(t_e, eeg_wave)
                plt.title(f'{dim} Top{rank} EEG 波形: Sample{sid}')
                plt.xlabel('Time (s)')
                plt.ylabel('Amplitude')
                plt.tight_layout()
                plt.savefig(os.path.join(output_dir, f'{dim}_Top{rank}_eeg_waveform.png'), dpi=300)
                plt.close()
                found = True
                break
            if found: break
        if not found:
            print(f'未找到对应 {dim} Top{rank} EEG 段')


C:\Users\33037\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


各象限最像的音乐及时间段：
  HVHA: 【音乐7.mp3】 第1段 (0s–60s)
  HVLA: 【音乐7.mp3】 第1段 (0s–60s)
  LVHA: 【音乐7.mp3】 第1段 (0s–60s)
  LVLA: 【音乐7.mp3】 第1段 (0s–60s)

--- Arousal 维度 Top2 ---
  Top1: 【音乐7.mp3】 第1段 (0s–60s)，次数 265
  Top2: 【音乐4.mp3】 第1段 (0s–60s)，次数 9

--- Valence 维度 Top2 ---
  Top1: 【音乐7.mp3】 第1段 (0s–60s)，次数 246
  Top2: 【音乐7.mp3】 第2段 (60s–120s)，次数 18


C:\Users\33037\AppData\Local\Temp\ipykernel_21944\328221068.py:111: UserWarning: Glyph 38899 (\N{CJK UNIFIED IDEOGRAPH-97F3}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21944\328221068.py:111: UserWarning: Glyph 20048 (\N{CJK UNIFIED IDEOGRAPH-4E50}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21944\328221068.py:111: UserWarning: Glyph 27874 (\N{CJK UNIFIED IDEOGRAPH-6CE2}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21944\328221068.py:111: UserWarning: Glyph 24418 (\N{CJK UNIFIED IDEOGRAPH-5F62}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21944\328221068.py:111: UserWarning: Glyph 27573 (\N{CJK UNIFIED IDEOGRAPH-6BB5}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21944\328221068.py:112: UserWarning: Glyph 38899 (\N{CJK UNIFIED IDEOGRAPH-97F3}) miss

In [3]:
import os
import pickle
import numpy as np
import librosa
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from collections import Counter, defaultdict

# —— 参数设置 ——#
mp3_dir    = r"C:\Users\33037\Desktop\判断音乐"
data_dir   = r"C:\Users\33037\Desktop\研究生论文\AT-DGNN-main\example"
output_dir = r"C:\Users\33037\Desktop\频谱图\音乐图片"
os.makedirs(output_dir, exist_ok=True)

# 函数定义

def cut_audio_segments(mp3_dir, seg_s=60, sr=22050):
    segs = []
    for fn in os.listdir(mp3_dir):
        if fn.startswith("音乐") and fn.endswith(".mp3"):
            y, _ = librosa.load(os.path.join(mp3_dir, fn), sr=sr)
            seg_len = seg_s * sr
            for idx in range(len(y) // seg_len):
                segs.append((fn, idx+1, y[idx*seg_len:(idx+1)*seg_len], sr))
    return segs


def extract_rms(y, frame_length=1024, hop_length=512):
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    return (rms - rms.mean()) / (rms.std() + 1e-6)


def quadrant_label(val, aro, v_th=0.5, a_th=0.5):
    if val >= v_th and aro >= a_th:
        return "HVHA"
    elif val >= v_th and aro <  a_th:
        return "HVLA"
    elif val <  v_th and aro >= a_th:
        return "LVHA"
    else:
        return "LVLA"

# 1. 切分音乐段
music_segs = cut_audio_segments(mp3_dir)

# 2. 匹配数据
emo_choices = defaultdict(list)
for sid in range(1, 33):
    eeg_file = os.path.join(data_dir, f"sample_{sid}.dat")
    unp = pickle._Unpickler(open(eeg_file, 'rb'))
    unp.encoding = 'latin1'
    trial = unp.load()
    data, labels = trial['data'], trial['labels']
    for seg_idx, (seg, lbl) in enumerate(zip(data, labels), start=1):
        val, aro = float(lbl[0]), float(lbl[1])
        emo = quadrant_label(val, aro)
        eeg_wave = seg.mean(axis=0)
        eeg_rms  = extract_rms(eeg_wave)
        best_corr = -2.0
        best_music = (None, None)
        for m_fn, m_si, m_wave, _ in music_segs:
            mus_rms = extract_rms(m_wave)
            L = min(len(eeg_rms), len(mus_rms))
            corr, _ = pearsonr(eeg_rms[:L], mus_rms[:L])
            if corr > best_corr:
                best_corr = corr
                best_music = (m_fn, m_si)
        emo_choices[emo].append((sid, seg_idx, best_music[0], best_music[1]))

# 3. 统计众数及 Top2 推荐
emo_mode = {emo: Counter([(fn,si) for *_,fn,si in lst]).most_common(1)[0][0]
            for emo, lst in emo_choices.items()}
high_aro = [(fn,si) for *_,fn,si in emo_choices['HVHA']+emo_choices['LVHA']]
arousal_top2 = Counter(high_aro).most_common(2)
high_val = [(fn,si) for *_,fn,si in emo_choices['HVHA']+emo_choices['HVLA']]
valence_top2 = Counter(high_val).most_common(2)

# 4. 打印结果及对应 sample 列表
print("各象限最像的音乐及时间段：")
for emo, (mf, si) in emo_mode.items():
    print(f"  {emo}: 【{mf}】 第{si}段 ({(si-1)*60}s–{si*60}s)")

print("\n--- Arousal Top2 推荐 & 对应 Samples ---")
for rank, ((mf,si),cnt) in enumerate(arousal_top2, start=1):
    # 收集匹配到此段的 sample id
    sids = [sid for sid,seg,fn,si2 in emo_choices['HVHA']+emo_choices['LVHA']
            if fn==mf and si2==si]
    print(f"  Top{rank}: 【{mf}】 第{si}段, 出现{cnt}次, Samples: {sids}")

print("\n--- Valence Top2 推荐 & 对应 Samples ---")
for rank, ((mf,si),cnt) in enumerate(valence_top2, start=1):
    sids = [sid for sid,seg,fn,si2 in emo_choices['HVHA']+emo_choices['HVLA']
            if fn==mf and si2==si]
    print(f"  Top{rank}: 【{mf}】 第{si}段, 出现{cnt}次, Samples: {sids}")

# 5. 绘制 Top1 & Top2 音乐波形
for dim, top2 in [('Arousal', arousal_top2), ('Valence', valence_top2)]:
    for rank, ((mf,si),_) in enumerate(top2, start=1):
        waveform, fs = next((w,sr) for fn,idx,w,sr in music_segs if fn==mf and idx==si)
        t = np.linspace(0, len(waveform)/fs, num=len(waveform))
        plt.figure(figsize=(10,2)); plt.plot(t, waveform)
        plt.title(f'{dim} Top{rank} 音乐波形: {mf} 段{si}')
        plt.xlabel('Time (s)'); plt.ylabel('Amplitude')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{dim}_Top{rank}_music_waveform.png'), dpi=300)
        plt.close()

# 6. 绘制所有 sample Top1 EEG 波形
for dim, top2 in [('Arousal', arousal_top2), ('Valence', valence_top2)]:
    mf, si = top2[0][0]
    pool = emo_choices['HVHA']+emo_choices['LVHA'] if dim=='Arousal' else emo_choices['HVHA']+emo_choices['HVLA']
    for sid, seg_idx, fn, si2 in pool:
        if fn!=mf or si2!=si: continue
        trial_unc = pickle._Unpickler(open(os.path.join(data_dir, f'sample_{sid}.dat'),'rb'))
        trial_unc.encoding='latin1'; trial = trial_unc.load()
        eeg_wave = trial['data'][seg_idx-1].mean(axis=0)
        sr_eeg = trial.get('fs',1000)
        t_e = np.linspace(0, len(eeg_wave)/sr_eeg, num=len(eeg_wave))
        plt.figure(figsize=(10,2)); plt.plot(t_e, eeg_wave)
        plt.title(f'{dim} Top1 EEG 波形: Sample{sid} 段{seg_idx}')
        plt.xlabel('Time (s)'); plt.ylabel('Amplitude')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{dim}_Top1_EEG_S{sid}_Seg{seg_idx}.png'), dpi=300)
        plt.close()


各象限最像的音乐及时间段：
  HVHA: 【音乐7.mp3】 第1段 (0s–60s)
  HVLA: 【音乐7.mp3】 第1段 (0s–60s)
  LVHA: 【音乐7.mp3】 第1段 (0s–60s)
  LVLA: 【音乐7.mp3】 第1段 (0s–60s)

--- Arousal Top2 推荐 & 对应 Samples ---
  Top1: 【音乐7.mp3】 第1段, 出现263次, Samples: [1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 10, 10, 10, 10, 10, 11, 12, 12, 12, 12, 12, 13, 13, 13, 13, 13, 14, 14, 14, 14, 14, 15, 15, 15, 15, 15, 16, 16, 16, 16, 16, 17, 17, 17, 17, 17, 18, 18, 18, 18, 18, 20, 20, 20, 21, 21, 21, 21, 22, 22, 22, 22, 23, 23, 23, 23, 24, 24, 25, 25, 28, 28, 28, 28, 28, 29, 29, 29, 29, 29, 30, 30, 30, 30, 30, 31, 31, 31, 32, 32, 32, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 10, 10, 10, 10, 10, 11, 12, 12, 12, 12, 12, 13, 13, 13, 13, 13, 14, 14, 14, 14, 14, 15, 15, 15, 15, 15, 16, 16, 16, 16, 16, 17, 17, 17, 17, 17, 18, 18, 18, 18, 18, 19, 19, 19, 19, 20, 20, 20, 20, 21, 21, 

C:\Users\33037\AppData\Local\Temp\ipykernel_21944\2237844151.py:104: UserWarning: Glyph 38899 (\N{CJK UNIFIED IDEOGRAPH-97F3}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21944\2237844151.py:104: UserWarning: Glyph 20048 (\N{CJK UNIFIED IDEOGRAPH-4E50}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21944\2237844151.py:104: UserWarning: Glyph 27874 (\N{CJK UNIFIED IDEOGRAPH-6CE2}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21944\2237844151.py:104: UserWarning: Glyph 24418 (\N{CJK UNIFIED IDEOGRAPH-5F62}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21944\2237844151.py:104: UserWarning: Glyph 27573 (\N{CJK UNIFIED IDEOGRAPH-6BB5}) missing from current font.
  plt.tight_layout()
C:\Users\33037\AppData\Local\Temp\ipykernel_21944\2237844151.py:105: UserWarning: Glyph 38899 (\N{CJK UNIFIED IDEOGRAPH-97F3}

In [1]:
#模型参数生成波形图
import os
import numpy as np
import matplotlib.pyplot as plt

# —— 路径配置 —— #
# 指向你的 param_log.txt 文件
PARAM_LOG = r"C:\Users\33037\Desktop\画图+波形图参数\save\param_log.txt"
# 波形图保存目录
OUTPUT_DIR = r"C:\Users\33037\Desktop\模型参数的波形图"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# —— 读取模型参数日志 —— #
param_lines = []
with open(PARAM_LOG, 'r', encoding='utf-8') as f:
    for line in f:
        if 'PARAMS:' in line:
            _, rest = line.split('PARAMS:')
            vals = [float(x) for x in rest.strip().split(',')]
            param_lines.append(vals)

# 转为 NumPy 数组，shape = (num_runs, num_params)
param_arr = np.array(param_lines)

# —— 绘制并保存每一次 run 的参数波形 —— #
for run_idx, vec in enumerate(param_arr, start=1):
    plt.figure(figsize=(10, 2))
    plt.plot(np.arange(len(vec)), vec, linewidth=0.8)
    plt.title(f'Run {run_idx} — Model Parameter Waveform')
    plt.xlabel('Parameter Index')
    plt.ylabel('Weight Value')
    plt.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, f'param_waveform_run_{run_idx}.png')
    plt.savefig(out_path, dpi=300)
    plt.close()

print(f"已生成 {param_arr.shape[0]} 张参数波形图，保存在：{OUTPUT_DIR}")


已生成 2 张参数波形图，保存在：C:\Users\33037\Desktop\模型参数的波形图


In [1]:
"""
param_vs_music.py
把 param_log.txt 中的参数（或直接给定的一行参数）：
 - 解析为向量
 - 标准化、平滑、插值到音乐 RMS 的帧数
 - 绘制三张图：参数原始波形、音乐 RMS 波形、二者重叠对比
 - 计算并输出 MSE 与 Pearson

依赖：numpy, librosa, scipy, matplotlib
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
from scipy.signal import savgol_filter
from scipy.stats import pearsonr

# ========== 配置 ==========

# 或者直接指定 param_log 文件路径并读取最后一行：
PARAM_LOG = r"C:\Users\33037\Desktop\画图+波形图参数\save\param_log.txt"

# 音乐文件路径（指向你要对齐的那首音乐）
MUSIC_FILE = r"C:\Users\33037\Desktop\判断音乐\音乐7.mp3"
MUSIC_START = 0        # 秒，起始位置
MUSIC_DUR = 60         # 秒，要比较的片段长度

# 输出目录
OUT_DIR = r"C:\Users\33037\Desktop\模型参数的波形图\param_vs_music"
os.makedirs(OUT_DIR, exist_ok=True)

# 参数预处理设置（可调）
SMOOTH_WINDOW = 11    # savgol_filter window length (must be odd)
SMOOTH_POLY = 3       # savgol_filter polynomial order
NORMALIZE = True      # 是否做 z-score 标准化
# 音乐 RMS 帧参数
FRAME_LENGTH = 2048
HOP_LENGTH = 512

# ========== 读取参数向量 ==========
def load_param_vector(param_log_path=None, param_line_str=None):
    if param_line_str is None and param_log_path is not None:
        # 读取文件最后一行包含 PARAMS: 的行（也可以改为读取多行）
        vecs = []
        with open(param_log_path, 'r', encoding='utf-8') as f:
            for line in f:
                if 'PARAMS:' in line:
                    rest = line.split('PARAMS:')[1].strip()
                    try:
                        vals = [float(x) for x in rest.split(',') if x.strip() != ""]
                        vecs.append(vals)
                    except:
                        pass
        if len(vecs) == 0:
            raise RuntimeError("param_log.txt 中未找到 PARAMS 行")
        # 返回最后一组（或你可以返回所有组）
        return np.array(vecs[-1], dtype=float)
    elif param_line_str is not None:
        vals = [float(x) for x in param_line_str.strip().split(',') if x.strip() != ""]
        return np.array(vals, dtype=float)
    else:
        raise ValueError("需要 param_log_path 或 param_line_str")

param_vec = load_param_vector(param_log_path=PARAM_LOG, param_line_str=None)
N = len(param_vec)
print(f"loaded param vector length: {N}")

# ========== 读取音乐并计算 RMS ==========
y, sr = librosa.load(MUSIC_FILE, sr=None)
s0 = int(MUSIC_START * sr)
s1 = min(len(y), s0 + int(MUSIC_DUR * sr))
y_clip = y[s0:s1]
music_rms = librosa.feature.rms(y=y_clip, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]
M = len(music_rms)
print(f"music rms frames: {M}, music sr: {sr}")

# 标准化函数
def zscore(x):
    return (x - np.mean(x)) / (np.std(x) + 1e-9)

# ========== 处理参数向量：去噪/平滑/标准化/插值 ==========
# 1) 标准化（先标准化再平滑或先平滑再标准化都可以，常见先标准化）
vec = param_vec.astype(float)
if NORMALIZE:
    vec = zscore(vec)

# 2) 平滑：Savitzky-Golay -> 能保留形状但去噪
# 检查窗口长度要求为奇数且 <= len(vec)
w = SMOOTH_WINDOW if SMOOTH_WINDOW % 2 == 1 else SMOOTH_WINDOW + 1
if w >= len(vec):
    w = len(vec) - 1 if (len(vec) - 1) % 2 == 1 else len(vec) - 2
    if w < 3:
        w = 3
try:
    vec_smooth = savgol_filter(vec, window_length=w, polyorder=SMOOTH_POLY)
except Exception as e:
    # 如果 fail，退回不平滑
    print("savgol_filter fail:", e)
    vec_smooth = vec

# 3) 插值 / 重采样到音乐帧数 M
t_param = np.linspace(0.0, 1.0, num=len(vec_smooth))
t_music = np.linspace(0.0, 1.0, num=M)
param_resampled = np.interp(t_music, t_param, vec_smooth)

# 4) 再次标准化（与music同尺度）
if NORMALIZE:
    param_resampled = zscore(param_resampled)
music_rms_z = zscore(music_rms)

# ========== 计算相似度指标 ==========
mse = float(np.mean((param_resampled - music_rms_z)**2))
try:
    pearson_r, pval = pearsonr(param_resampled, music_rms_z)
except Exception:
    pearson_r, pval = float('nan'), float('nan')
print(f"MSE={mse:.6e}, Pearson={pearson_r:.4f}")

# ========== 绘图 ==========
# 1) 参数原始向量（标准化后）
plt.figure(figsize=(12,2))
plt.plot(np.arange(len(param_vec)), zscore(param_vec), linewidth=0.8)
plt.title("Parameter vector (z-score, raw indices)")
plt.xlabel("Parameter index")
plt.ylabel("z-score")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "param_raw_z.png"), dpi=300)
plt.close()

# 2) 参数平滑并重采样后的波形（与音乐帧对齐）
time_music = np.linspace(0, MUSIC_DUR, num=M)
plt.figure(figsize=(12,2))
plt.plot(time_music, param_resampled, label="Param (resampled, z)")
plt.plot(time_music, music_rms_z, label="Music RMS (z)", alpha=0.9)
plt.legend()
plt.title(f"Param vs Music — MSE={mse:.6e}, Pearson={pearson_r:.3f}")
plt.xlabel("Time (s)")
plt.ylabel("Normalized amplitude")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "param_vs_music_overlay.png"), dpi=300)
plt.close()

# 3) 单独绘音乐RMS（便于对比）
plt.figure(figsize=(12,2))
plt.plot(time_music, music_rms_z)
plt.title("Music RMS (z-score)")
plt.xlabel("Time (s)")
plt.ylabel("Normalized RMS")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "music_rms_z.png"), dpi=300)
plt.close()

# 4) 保存数值与结果
np.savetxt(os.path.join(OUT_DIR, "param_resampled.txt"), param_resampled, fmt="%.6f")
with open(os.path.join(OUT_DIR, "param_vs_music_metrics.txt"), "w", encoding="utf-8") as wf:
    wf.write(f"MSE={mse:.6e}\nPearson={pearson_r:.6f}\npearson_pval={pval}\n")
print("Saved plots and metrics to", OUT_DIR)


loaded param vector length: 1024


C:\Users\33037\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


music rms frames: 5168, music sr: 44100
MSE=2.080507e+00, Pearson=-0.0403
Saved plots and metrics to C:\Users\33037\Desktop\模型参数的波形图\param_vs_music


In [3]:
# process_param_log.py
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import librosa
from scipy.signal import savgol_filter
from scipy.stats import pearsonr

# ========== 配置区（请修改这三项） ==========
PARAM_LOG = r"C:\Users\33037\Desktop\画图+波形图参数\save\param_log.txt"
MUSIC_FILE = r"C:\Users\33037\Desktop\判断音乐\音乐7.mp3"
OUT_ROOT = r"C:\Users\33037\Desktop\模型参数的波形图\param_vs_music"
# ============================================

os.makedirs(OUT_ROOT, exist_ok=True)

# 平滑与音乐RMS设置（可调整）
SMOOTH_WINDOW = 11   # must be odd
SMOOTH_POLY = 3
NORMALIZE = True
FRAME_LENGTH = 2048
HOP_LENGTH = 512

# 解析 param_log.txt 中所有 SUB/FOLD 条目
entries = []  # list of dicts: {'sub':int,'fold':int,'vec':np.array}
pat = re.compile(r"SUB\s*:\s*(\d+)\s*FOLD\s*:\s*(\d+).*?PARAMS\s*:\s*(.+)", flags=re.IGNORECASE)
with open(PARAM_LOG, 'r', encoding='utf-8') as f:
    for line in f:
        m = pat.search(line)
        if m:
            sub = int(m.group(1))
            fold = int(m.group(2))
            params_str = m.group(3).strip()
            # 有时行后续有其他，取到行末
            # 解析数值
            vals = []
            for tok in params_str.split(','):
                tok = tok.strip()
                if tok == '':
                    continue
                try:
                    vals.append(float(tok))
                except:
                    # 如果遇到非数字（断行等），跳过
                    pass
            if len(vals) > 0:
                entries.append({'sub': sub, 'fold': fold, 'vec': np.array(vals, dtype=float)})

if len(entries) == 0:
    raise RuntimeError("未在 param_log.txt 中找到任何带 PARAMS 的行。请检查文件路径与格式。")

print(f"解析到 {len(entries)} 条 PARAMS 条目。示例：", entries[0]['sub'], entries[0]['fold'], "长度:", len(entries[0]['vec']))

# 读取音乐并计算 RMS（一次读入，所有组共用）
y, sr = librosa.load(MUSIC_FILE, sr=None)
music_dur_seconds = len(y) / sr
print(f"读取音乐: sr={sr}, 时长={music_dur_seconds:.2f}s")
# 取默认前 60s 或整首（如果短于60s）
MUSIC_START = 0
MUSIC_DUR = min(60, int(music_dur_seconds))
s0 = int(MUSIC_START * sr)
s1 = min(len(y), s0 + int(MUSIC_DUR * sr))
y_clip = y[s0:s1]
music_rms = librosa.feature.rms(y=y_clip, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]
M = len(music_rms)
print("Music RMS frames:", M)

def zscore(x):
    return (x - np.mean(x)) / (np.std(x) + 1e-9)

# 主循环：逐条处理
for ent in entries:
    sub = ent['sub']
    fold = ent['fold']
    vec = ent['vec'].astype(float)
    out_dir = os.path.join(OUT_ROOT, f"SUB_{sub}_FOLD_{fold}")
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n处理 SUB:{sub} FOLD:{fold}，参数长度={len(vec)}，输出到 {out_dir}")

    # 1) param_raw_z.png (raw param standardized)
    if NORMALIZE:
        vec_z = zscore(vec)
    else:
        vec_z = vec.copy()
    plt.figure(figsize=(12,2))
    plt.plot(np.arange(len(vec_z)), vec_z, linewidth=0.8)
    plt.title(f"SUB{sub}_FOLD{fold} — Parameter vector (z-score)")
    plt.xlabel("Parameter index")
    plt.ylabel("z-score")
    plt.tight_layout()
    path1 = os.path.join(out_dir, "param_raw_z.png")
    plt.savefig(path1, dpi=300); plt.close()

    # 2) 平滑
    w = SMOOTH_WINDOW if SMOOTH_WINDOW % 2 == 1 else SMOOTH_WINDOW + 1
    if w >= len(vec_z):
        # set to max odd < len(vec_z)
        w = len(vec_z) - 1
        if w % 2 == 0:
            w -= 1
        if w < 3:
            w = 3
    try:
        vec_smooth = savgol_filter(vec_z, window_length=w, polyorder=SMOOTH_POLY)
    except Exception as e:
        print("savgol_filter failed, using unfiltered:", e)
        vec_smooth = vec_z

    # 3) 插值到音乐帧数 M（把参数沿归一化时间轴重采样）
    t_param = np.linspace(0.0, 1.0, num=len(vec_smooth))
    t_music = np.linspace(0.0, 1.0, num=M)
    param_resampled = np.interp(t_music, t_param, vec_smooth)

    # 标准化（与 music 同尺度）
    param_resampled_z = zscore(param_resampled)
    music_rms_z = zscore(music_rms)

    # 4) 计算 MSE 与 Pearson
    mse = float(np.mean((param_resampled_z - music_rms_z)**2))
    try:
        pearson_r, pval = pearsonr(param_resampled_z, music_rms_z)
    except Exception:
        pearson_r, pval = float('nan'), float('nan')
    print(f"  MSE={mse:.6e}, Pearson={pearson_r:.4f}")

    # 5) 保存 param_resampled.txt
    np.savetxt(os.path.join(out_dir, "param_resampled.txt"), param_resampled_z, fmt="%.6f")

    # 6) 保存 metrics
    with open(os.path.join(out_dir, "param_vs_music_metrics.txt"), "w", encoding="utf-8") as wf:
        wf.write(f"MSE={mse:.6e}\n")
        wf.write(f"Pearson={pearson_r:.6f}\n")
        wf.write(f"Pearson_pval={pval}\n")

    # 7) 保存单独音乐RMS图
    time_music = np.linspace(0, MUSIC_DUR, num=M)
    plt.figure(figsize=(12,2))
    plt.plot(time_music, music_rms_z, label="Music RMS (z)")
    plt.title(f"SUB{sub}_FOLD{fold} — Music RMS (z)")
    plt.xlabel("Time (s)"); plt.ylabel("z-score")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "music_rms_z.png"), dpi=300)
    plt.close()

    # 8) 保存 overlay 图（param vs music）
    plt.figure(figsize=(12,2))
    plt.plot(time_music, param_resampled_z, label="Param (resampled, z)")
    plt.plot(time_music, music_rms_z, label="Music RMS (z)", alpha=0.9)
    plt.legend()
    plt.title(f"SUB{sub}_FOLD{fold} — Param vs Music (MSE={mse:.6e}, r={pearson_r:.3f})")
    plt.xlabel("Time (s)"); plt.ylabel("z-score")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "param_vs_music_overlay.png"), dpi=300)
    plt.close()

print("\n全部处理完成。每组结果保存在:", OUT_ROOT)


解析到 2 条 PARAMS 条目。示例： 0 0 长度: 1024
读取音乐: sr=44100, 时长=416.66s
Music RMS frames: 5168

处理 SUB:0 FOLD:0，参数长度=1024，输出到 C:\Users\33037\Desktop\模型参数的波形图\param_vs_music\SUB_0_FOLD_0
  MSE=1.971017e+00, Pearson=0.0145

处理 SUB:0 FOLD:1，参数长度=1024，输出到 C:\Users\33037\Desktop\模型参数的波形图\param_vs_music\SUB_0_FOLD_1
  MSE=2.080507e+00, Pearson=-0.0403

全部处理完成。每组结果保存在: C:\Users\33037\Desktop\模型参数的波形图\param_vs_music


In [1]:
#生成音乐RMS文件
# make_music_envelope.py
import librosa
import numpy as np
from scipy.signal import hilbert

MUSIC_FILE = r"C:\Users\33037\Desktop\判断音乐\音乐7.mp3"
OUT_NPY = r"C:\Users\33037\Desktop\判断音乐\RMS文件\music7_envelope.npy"
FRAME_LENGTH = 2048
HOP_LENGTH = 512

y, sr = librosa.load(MUSIC_FILE, sr=None)
# --- 方法1: 短时 RMS（推荐） ---
rms = librosa.feature.rms(y=y, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]  # shape (M,)
# --- 方法2: Hilbert 包络（逐样本）然后帧化/降采样（示例） ---
analytic = hilbert(y)
env = np.abs(analytic)  # 每个采样点的包络
# 将 env 按 hop_length 抽帧取平均以得到与 rms 类似的帧率：
frames = []
for i in range(0, len(env) - FRAME_LENGTH + 1, HOP_LENGTH):
    frames.append(env[i:i+FRAME_LENGTH].mean())
env_frames = np.array(frames, dtype=np.float32)

# 你可以选择保存 rms 或 env_frames（两者通常相近）
np.save(OUT_NPY, rms.astype(np.float32))
print("Saved RMS len:", rms.shape[0])


C:\Users\33037\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


Saved RMS len: 35889


In [3]:
#生成损失函数
"""
compute_param_music_losses.py

离线从 param_log.txt 与预计算的音乐 RMS（.npy/.npz）生成一系列损失/相似度指标，
并为每个 SUB/FOLD 保存：param_resampled.txt、overlay 图、以及汇总 CSV：results.csv

依赖：numpy, scipy, matplotlib
可选依赖（若要 DTW）：fastdtw
"""

import os, re, sys
import numpy as np
from scipy.signal import savgol_filter, stft
from scipy.stats import pearsonr
from scipy.spatial.distance import cosine
import matplotlib.pyplot as plt

# 可选 DTW
try:
    from fastdtw import fastdtw
    from scipy.spatial.distance import euclidean
    HAS_DTW = True
except Exception:
    HAS_DTW = False

# ========== 配置区 ==========
PARAM_LOG = r"C:\Users\33037\Desktop\画图+波形图参数\save\param_log.txt"
# MUSIC_RMS_PATH 可以是 .npy (一维) 或 .npz (rms / rms_z)
MUSIC_RMS_PATH = r"C:\Users\33037\Desktop\判断音乐\RMS文件\music7_envelope.npy"
OUT_ROOT = r"C:\Users\33037\Desktop\模型参数的波形图\param_losses"
# 平滑与 STFT 设置
SMOOTH_WINDOW = 11   # savgol window (odd)
SMOOTH_POLY = 3
FRAME_LENGTH = 2048
HOP_LENGTH = 512
# 标准化方式： 'z' or None
NORMALIZE = 'z'
# ===========================

os.makedirs(OUT_ROOT, exist_ok=True)

# --- helper functions ---
def parse_param_log(path):
    """解析 param_log.txt，返回 list of dict {'sub':int,'fold':int,'vec':np.array}"""
    pat = re.compile(r"SUB\s*[:=]\s*(\d+)\s+FOLD\s*[:=]\s*(\d+).*?PARAMS\s*[:=]\s*(.+)", flags=re.IGNORECASE)
    entries = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            m = pat.search(line)
            if m:
                sub = int(m.group(1)); fold = int(m.group(2))
                params_str = m.group(3).strip()
                # 有可能参数跨行，简单处理：去掉尾部非数字/逗号字符
                # split by comma
                toks = [t.strip() for t in params_str.split(',') if t.strip() != '']
                vals = []
                for t in toks:
                    try:
                        vals.append(float(t))
                    except:
                        # 忽略解析失败的 token
                        pass
                if len(vals) > 0:
                    entries.append({'sub': sub, 'fold': fold, 'vec': np.array(vals, dtype=float)})
    return entries

def load_music_rms(path):
    """加载 .npy 或 .npz , 返回 1D numpy array（若存在 rms_z 优先）"""
    if path.endswith('.npy'):
        arr = np.load(path)
        return arr.astype(np.float32)
    elif path.endswith('.npz'):
        data = np.load(path)
        if 'rms_z' in data:
            return data['rms_z'].astype(np.float32)
        elif 'rms' in data:
            return data['rms'].astype(np.float32)
        else:
            # 取第一个数组
            keys = list(data.keys())
            return data[keys[0]].astype(np.float32)
    else:
        raise RuntimeError("支持 .npy 或 .npz 文件")

def zscore(x):
    x = np.array(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)

def param_to_timeseries(vec, M, smooth_window=11, poly=3, normalize='z'):
    """把参数向量 -> (可能)标准化 -> 平滑 -> 插值到长度 M -> 返回 1D array"""
    v = np.array(vec, dtype=float)
    if normalize == 'z':
        v = zscore(v)
    # 平滑（if len small, skip)
    w = smooth_window if smooth_window % 2 == 1 else smooth_window+1
    if len(v) >= w and w >= 3:
        try:
            v_sm = savgol_filter(v, window_length=w, polyorder=poly)
        except Exception:
            v_sm = v
    else:
        v_sm = v
    # 插值到 M
    t_param = np.linspace(0, 1, num=len(v_sm))
    t_target = np.linspace(0, 1, num=M)
    res = np.interp(t_target, t_param, v_sm)
    if normalize == 'z':
        res = zscore(res)
    return res

def spectral_mse(a, b, fs=1.0, nperseg=256):
    """计算两个1D信号的 STFT 幅度谱的 MSE（归一化）"""
    # 使用 scipy.signal.stft
    f1, t1, Z1 = stft(a, fs=fs, nperseg=nperseg)
    f2, t2, Z2 = stft(b, fs=fs, nperseg=nperseg)
    # 可能 t1 != t2 长度不同，插值时间轴到最小长度
    T = min(Z1.shape[1], Z2.shape[1])
    mag1 = np.abs(Z1)[:, :T]
    mag2 = np.abs(Z2)[:, :T]
    mse = np.mean((mag1 - mag2)**2)
    return mse

# --- 主流程 ---
entries = parse_param_log(PARAM_LOG)
if len(entries) == 0:
    print("没有解析到 PARAMS 条目，请检查 param_log 文件格式。")
    sys.exit(1)
music_rms = load_music_rms(MUSIC_RMS_PATH)
M = len(music_rms)
print(f"找到 {len(entries)} 条参数记录，音乐 RMS 长度 M={M}")

# 若音乐 RMS 不是 z-score，则这里做 z-score（统一）
if NORMALIZE == 'z':
    music_ts = zscore(music_rms)
else:
    music_ts = music_rms.copy()

results = []
for ent in entries:
    sub = ent['sub']; fold = ent['fold']; vec = ent['vec']
    out_dir = os.path.join(OUT_ROOT, f"SUB_{sub}_FOLD_{fold}")
    os.makedirs(out_dir, exist_ok=True)
    # param -> timeseries
    p_ts = param_to_timeseries(vec, M, smooth_window=SMOOTH_WINDOW, poly=SMOOTH_POLY, normalize=NORMALIZE)
    # 计算指标
    mse = float(np.mean((p_ts - music_ts)**2))
    # Pearson
    try:
        pr, pval = pearsonr(p_ts, music_ts)
    except Exception:
        pr, pval = float('nan'), float('nan')
    corr_loss = 1.0 - pr
    # Cosine distance (scipy cosine returns distance 0..2 for normalized vectors)
    try:
        cosd = float(cosine(p_ts, music_ts))  # smaller = more similar; cosine distance in [0,2]
    except Exception:
        cosd = float('nan')
    # spectral mse
    try:
        spec_mse = float(spectral_mse(p_ts, music_ts, fs=1.0, nperseg=256))
    except Exception:
        spec_mse = float('nan')
    # optional DTW
    if HAS_DTW:
        try:
            dtw_dist, _ = fastdtw(p_ts, music_ts, dist=euclidean)
        except Exception:
            dtw_dist = float('nan')
    else:
        dtw_dist = float('nan')

    # 保存 param_resampled 配置与 overlay 图
    np.savetxt(os.path.join(out_dir, "param_resampled.txt"), p_ts, fmt="%.6f")
    # overlay plot
    t = np.linspace(0, M, num=M)  # 单位为帧；如果你需要秒，可以乘以 hop_length/sr
    plt.figure(figsize=(10,2.5))
    plt.plot(t, p_ts, label='Param (resampled)', linewidth=1)
    plt.plot(t, music_ts, label='Music RMS', linewidth=1, alpha=0.9)
    plt.legend(loc='upper right')
    plt.title(f"SUB{sub}_FOLD{fold} — mse={mse:.2e}, pearson={pr:.3f}")
    plt.xlabel("Frame")
    plt.ylabel("z-score")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "param_vs_music_overlay.png"), dpi=200)
    plt.close()

    # 保存单条 metrics 到 results 列表与单文件
    res = {
        'sub': sub, 'fold': fold,
        'mse': mse, 'pearson': pr, 'pearson_p': pval,
        'corr_loss': corr_loss, 'cosine_dist': cosd,
        'spectral_mse': spec_mse, 'dtw': dtw_dist
    }
    results.append(res)

# 保存汇总 CSV
import csv
csv_path = os.path.join(OUT_ROOT, "results_summary.csv")
with open(csv_path, 'w', newline='', encoding='utf-8') as cf:
    writer = csv.DictWriter(cf, fieldnames=['sub','fold','mse','pearson','pearson_p','corr_loss','cosine_dist','spectral_mse','dtw'])
    writer.writeheader()
    for r in results:
        writer.writerow(r)

print("Done. Results saved to:", OUT_ROOT)
if not HAS_DTW:
    print("If you want DTW, install fastdtw (pip install fastdtw).")


找到 2 条参数记录，音乐 RMS 长度 M=35889
Done. Results saved to: C:\Users\33037\Desktop\模型参数的波形图\param_losses
If you want DTW, install fastdtw (pip install fastdtw).


In [1]:
# results_summary.csv 生成 combined_loss
import pandas as pd
import numpy as np
out_csv = r"C:\Users\33037\Desktop\模型参数的波形图\param_losses\results_summary.csv"
df = pd.read_csv(out_csv)

# 简单策略：combined = mse + 0.1*(1-pearson)
df['corr_loss'] = 1.0 - df['pearson']
df['combined_simple'] = df['mse'] + 0.1 * df['corr_loss']

# 归一化策略：min-max normalize each metric across all rows, then weighted sum
cols_to_norm = ['mse', 'corr_loss', 'spectral_mse', 'cosine_dist']
for c in cols_to_norm:
    if c in df.columns:
        mn = df[c].min()
        mx = df[c].max()
        if mx - mn > 0:
            df[c + '_norm'] = (df[c] - mn) / (mx - mn)
        else:
            df[c + '_norm'] = 0.0

# weighted combination (tune weights)
w = {'mse_norm': 0.5, 'corr_loss_norm': 0.3, 'spectral_mse_norm': 0.2}
df['combined_norm'] = 0.0
for k, wt in w.items():
    if k in df.columns:
        df['combined_norm'] += wt * df[k]

# 保存新 CSV
df.to_csv(out_csv.replace('.csv', '_with_combined.csv'), index=False)
print("Saved new CSV with combined loss.")


Saved new CSV with combined loss.


In [1]:
#横坐标是轮数epoch的版本
"""
plot_param_training_trend.py

作用：
1) 解析 param_log.txt（按行读取，每行若含 "SUB:x FOLD:y PARAMS: ..." 形式）
2) 若行中含有 "EP:" / "EPOCH:" / "Epoch" / "step" 等，会尝试解析为 epoch/step；否则按出现顺序计数为 index
3) 将参数向量平滑并插值到音乐 RMS 的帧数 M（music_rms.npy / .npz）
4) 计算每条记录的 MSE 与 Pearson，并画出随 index/epoch 的趋势图
5) 另存并绘制起始/中期/末期的 overlay 波形图（参数 vs 音乐）

输出：
- results_summary.csv (index,sub,fold,epoch,mse,pearson)
- trend_mse.png （横轴：index/epoch，纵轴：MSE）
- trend_pearson.png （横轴：index/epoch，纵轴：Pearson）
- overlay_first.png / overlay_mid.png / overlay_last.png（时间序列对比图）
- 各记录的 param_resampled.txt 保存到 OUT_DIR/SUB_x_FOLD_y_IDX/ 下

依赖： numpy, scipy, matplotlib
"""
import os, re, csv
import numpy as np
from scipy.signal import savgol_filter
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# ========== 配置 ==========
PARAM_LOG = r"C:\Users\33037\Desktop\画图与波形图参数\save\param_log.txt"
MUSIC_RMS_PATH = r"C:\Users\33037\Desktop\论文\论文数据\判断音乐\RMS文件\music7_envelope.npy"  # 或 .npz 或 .npy
OUT_DIR = r"C:\Users\33037\Desktop\论文\论文数据\模型参数的波形图"
SMOOTH_WINDOW = 11   # savgol window length (odd)
SMOOTH_POLY = 3
NORMALIZE = 'z'      # 对参数与音乐做 z-score（建议）
# =======================

os.makedirs(OUT_DIR, exist_ok=True)

# --------- 辅助函数 ----------
def load_music_rms(path):
    if path.endswith('.npy'):
        arr = np.load(path)
        return arr.astype(np.float32)
    elif path.endswith('.npz'):
        d = np.load(path)
        # 优先取 rms_z, 再 rms
        if 'rms_z' in d: return d['rms_z'].astype(np.float32)
        if 'rms' in d: return d['rms'].astype(np.float32)
        # 否则取第一个数组
        key = list(d.keys())[0]
        return d[key].astype(np.float32)
    else:
        raise RuntimeError("请提供 .npy 或 .npz 的 RMS 文件")

def zscore(x):
    x = np.array(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)

def parse_param_log(path):
    """
    返回 list of dict:
      {'line_idx':int, 'sub':int-or-None, 'fold':int-or-None, 'epoch':int-or-None, 'vec':np.array}
    """
    pat = re.compile(r"SUB\s*[:=]\s*(\d+)\s*FOLD\s*[:=]\s*(\d+).*?PARAMS\s*[:=]\s*(.+)", flags=re.IGNORECASE)
    # 也尝试搜索 epoch 信息（EP: 或 epoch= 或 Epoch）
    ep_pat = re.compile(r"(?:EP|Epoch|EPOCH|epoch|step)\s*[:=]?\s*(\d+)", flags=re.IGNORECASE)
    entries = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            m = pat.search(line)
            if not m:
                # 兼容：有些行可能只写 PARAMS: ... 没 SUB/FOLD，尝试只找 PARAMS:
                m2 = re.search(r"PARAMS\s*[:=]\s*(.+)", line, flags=re.IGNORECASE)
                if not m2:
                    continue
                sub = None; fold = None
                params_str = m2.group(1).strip()
            else:
                sub = int(m.group(1)); fold = int(m.group(2))
                params_str = m.group(3).strip()
            # 尝试解析 epoch
            ep_m = ep_pat.search(line)
            epoch = int(ep_m.group(1)) if ep_m else None

            # parse numeric params (comma separated)
            toks = [t.strip() for t in params_str.split(',') if t.strip()!='']
            vals = []
            for t in toks:
                try:
                    vals.append(float(t))
                except:
                    # 如果遇到非数字，忽略
                    pass
            if len(vals) == 0:
                continue
            entries.append({'line_idx': i, 'sub': sub, 'fold': fold, 'epoch': epoch, 'vec': np.array(vals, dtype=float)})
    return entries

def param_to_ts(vec, M, smooth_window=11, poly=3, normalize='z'):
    v = np.array(vec, dtype=float)
    if normalize == 'z':
        v = zscore(v)
    # savgol
    w = smooth_window if (smooth_window % 2 == 1) else smooth_window+1
    if len(v) >= w and w >= 3:
        try:
            v = savgol_filter(v, window_length=w, polyorder=poly)
        except Exception:
            pass
    # 插值
    t_src = np.linspace(0,1,num=len(v))
    t_tgt = np.linspace(0,1,num=M)
    res = np.interp(t_tgt, t_src, v)
    if normalize == 'z':
        res = zscore(res)
    return res

# ---------- 主流程 ----------
entries = parse_param_log(PARAM_LOG)
if len(entries) == 0:
    raise RuntimeError("未在 param_log.txt 中解析到任何 PARAMS 条目，请检查路径和文件格式。")

music_rms = load_music_rms(MUSIC_RMS_PATH)
if NORMALIZE == 'z':
    music_ts = zscore(music_rms)
else:
    music_ts = music_rms.copy()
M = len(music_ts)
print(f"解析到 {len(entries)} 条参数记录，音乐帧数 M={M}")

# 为每条记录计算 metrics（按文件中出现顺序）
rows = []
for idx, ent in enumerate(entries):
    vec = ent['vec']
    p_ts = param_to_ts(vec, M, SMOOTH_WINDOW, SMOOTH_POLY, normalize=NORMALIZE)
    mse = float(np.mean((p_ts - music_ts)**2))
    try:
        pr, pval = pearsonr(p_ts, music_ts)
    except Exception:
        pr, pval = float('nan'), float('nan')
    # 保存按记录
    rec = {
        'idx': idx, 'line_idx': ent['line_idx'],
        'sub': ent['sub'], 'fold': ent['fold'], 'epoch': ent['epoch'] if ent['epoch'] is not None else idx,
        'mse': mse, 'pearson': pr, 'pearson_p': pval
    }
    rows.append((rec, p_ts))
    # 保存 param_resampled
    out_subdir = os.path.join(OUT_DIR, f"record_{idx}_sub{ent['sub']}_fold{ent['fold']}")
    os.makedirs(out_subdir, exist_ok=True)
    np.savetxt(os.path.join(out_subdir, "param_resampled.txt"), p_ts, fmt="%.6f")

# 保存汇总 CSV（按文件顺序）
csv_path = os.path.join(OUT_DIR, "results_summary.csv")
with open(csv_path, 'w', newline='', encoding='utf-8') as cf:
    writer = csv.writer(cf)
    writer.writerow(['index','line_idx','sub','fold','epoch','mse','pearson','pearson_p'])
    for rec, _ in rows:
        writer.writerow([rec['idx'], rec['line_idx'], rec['sub'], rec['fold'], rec['epoch'], f"{rec['mse']:.6e}", f"{rec['pearson']:.6f}", rec['pearson_p']])

# 绘制趋势图：MSE（越低越好）与 Pearson（越高越好）
indices = [r[0]['epoch'] for r in rows]  # 若 epoch 存在则用 epoch，否则用记录索引
mse_vals = [r[0]['mse'] for r in rows]
pear_vals = [r[0]['pearson'] for r in rows]

plt.figure(figsize=(8,4))
plt.plot(indices, mse_vals, marker='o', linestyle='-')
plt.xlabel("epoch / record index")
plt.ylabel("MSE (param vs music)")
plt.title("Param vs Music — MSE trend (lower is better)")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "trend_mse.png"), dpi=300)
plt.close()

plt.figure(figsize=(8,4))
plt.plot(indices, pear_vals, marker='o', linestyle='-')
plt.xlabel("epoch / record index")
plt.ylabel("Pearson correlation (param vs music)")
plt.title("Param vs Music — Pearson trend (higher is better)")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "trend_pearson.png"), dpi=300)
plt.close()

# 画示例 overlay（first / mid / last）
def plot_overlay(p_ts, music_ts, out_png, title):
    T = len(p_ts)
    t = np.linspace(0, T-1, T)  # 横轴以帧编号表示；若要秒可乘 hop/sr
    plt.figure(figsize=(10,2.4))
    plt.plot(t, p_ts, label='Param (resampled)', linewidth=1)
    plt.plot(t, music_ts, label='Music RMS', linewidth=1, alpha=0.9)
    plt.legend()
    plt.xlabel("frame")
    plt.ylabel("z-score")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

first_ts = rows[0][1]
mid_ts = rows[len(rows)//2][1]
last_ts = rows[-1][1]
plot_overlay(first_ts, music_ts, os.path.join(OUT_DIR, "overlay_first.png"), "Overlay — first record")
plot_overlay(mid_ts, music_ts, os.path.join(OUT_DIR, "overlay_mid.png"), "Overlay — middle record")
plot_overlay(last_ts, music_ts, os.path.join(OUT_DIR, "overlay_last.png"), "Overlay — last record")

print("输出保存在：", OUT_DIR)
print("解释：")
print(" - overlay_xxx.png 的横轴为帧编号（frame），若需要秒请用 frame_index * (hop_length / sr)。纵轴为标准化后的幅值（z-score）。")
print(" - trend_mse.png 横轴为 epoch 或记录序号（取决于 param_log 是否包含 epoch 字段），纵轴为 MSE（越低越好）。")
print(" - trend_pearson.png 横轴同上，纵轴为 Pearson 相关系数（越高越好，接近 1 表示形状非常相似）。")


解析到 2 条参数记录，音乐帧数 M=35889
输出保存在： C:\Users\33037\Desktop\论文\论文数据\模型参数的波形图
解释：
 - overlay_xxx.png 的横轴为帧编号（frame），若需要秒请用 frame_index * (hop_length / sr)。纵轴为标准化后的幅值（z-score）。
 - trend_mse.png 横轴为 epoch 或记录序号（取决于 param_log 是否包含 epoch 字段），纵轴为 MSE（越低越好）。
 - trend_pearson.png 横轴同上，纵轴为 Pearson 相关系数（越高越好，接近 1 表示形状非常相似）。


In [7]:
# plot_param_training_trend_threeway.py
"""
Three-way plotting:
 - reads your param_log.txt (your created loss-based params)
 - reads orig_loss_param_log.txt (original loss converted to params)
 - reads music RMS (.npy/.npz)
 - computes MSE/Pearson of each param-record vs music, plots trends for both sets
 - creates overlay_{first,mid,last}.png showing three curves (param, orig_param, music)
"""
import os, re, csv
import numpy as np
from scipy.signal import savgol_filter
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# ========== 配置（请按需要修改路径） ==========
PARAM_LOG = r"C:\Users\33037\Desktop\画图与波形图三个曲线版\save\param_log.txt"
ORIG_PARAM_LOG = r"C:\Users\33037\Desktop\画图与波形图三个曲线版\save\orig_loss_param_log.txt"
MUSIC_RMS_PATH = r"C:\Users\33037\Desktop\论文\论文数据\判断音乐\RMS文件\music7_envelope.npy"
OUT_DIR = r"C:\Users\33037\Desktop\论文\论文数据\模型参数的波形图"
SMOOTH_WINDOW = 11   # savgol window length (odd)
SMOOTH_POLY = 3
NORMALIZE = 'z'      # 'z' or None
os.makedirs(OUT_DIR, exist_ok=True)
# =======================

# --------- 辅助函数 ----------
def load_music_rms(path):
    if path.endswith('.npy'):
        arr = np.load(path)
        return arr.astype(np.float32)
    elif path.endswith('.npz'):
        d = np.load(path)
        if 'rms_z' in d: return d['rms_z'].astype(np.float32)
        if 'rms' in d: return d['rms'].astype(np.float32)
        key = list(d.keys())[0]
        return d[key].astype(np.float32)
    else:
        raise RuntimeError("请提供 .npy 或 .npz 的 RMS 文件")

def zscore(x):
    x = np.array(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)

def parse_param_log(path):
    """
    返回 list of dict:
      {'line_idx':int, 'sub':int-or-None, 'fold':int-or-None, 'epoch':int-or-None, 'vec':np.array}
    """
    if not os.path.exists(path):
        return []
    pat = re.compile(r"SUB\s*[:=]\s*(\d+)\s*FOLD\s*[:=]\s*(\d+).*?PARAMS\s*[:=]\s*(.+)", flags=re.IGNORECASE)
    ep_pat = re.compile(r"(?:EP|Epoch|EPOCH|epoch|step)\s*[:=]?\s*(\d+)", flags=re.IGNORECASE)
    entries = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            m = pat.search(line)
            if not m:
                m2 = re.search(r"PARAMS\s*[:=]\s*(.+)", line, flags=re.IGNORECASE)
                if not m2:
                    continue
                sub = None; fold = None
                params_str = m2.group(1).strip()
            else:
                sub = int(m.group(1)); fold = int(m.group(2))
                params_str = m.group(3).strip()
            ep_m = ep_pat.search(line)
            epoch = int(ep_m.group(1)) if ep_m else None
            toks = [t.strip() for t in params_str.split(',') if t.strip()!='']
            vals = []
            for t in toks:
                try:
                    vals.append(float(t))
                except:
                    pass
            if len(vals) == 0:
                continue
            entries.append({'line_idx': i, 'sub': sub, 'fold': fold, 'epoch': epoch, 'vec': np.array(vals, dtype=float)})
    return entries

def param_to_ts(vec, M, smooth_window=11, poly=3, normalize='z'):
    v = np.array(vec, dtype=float)
    if normalize == 'z':
        v = zscore(v)
    w = smooth_window if (smooth_window % 2 == 1) else smooth_window+1
    if len(v) >= w and w >= 3:
        try:
            v = savgol_filter(v, window_length=w, polyorder=poly)
        except Exception:
            pass
    t_src = np.linspace(0,1,num=len(v))
    t_tgt = np.linspace(0,1,num=M)
    res = np.interp(t_tgt, t_src, v)
    if normalize == 'z':
        res = zscore(res)
    return res

# ---------- 主流程 ----------
# 1) 读取三份数据
entries_main = parse_param_log(PARAM_LOG)
entries_orig = parse_param_log(ORIG_PARAM_LOG)

if len(entries_main) == 0 and len(entries_orig) == 0:
    raise RuntimeError("两份 param_log 都未解析到任何条目，请检查路径。")

music_rms = load_music_rms(MUSIC_RMS_PATH)
music_ts = zscore(music_rms) if NORMALIZE == 'z' else music_rms.copy()
M = len(music_ts)
print(f"音乐帧数 M={M}. main entries: {len(entries_main)}, orig entries: {len(entries_orig)}")

# 2) 对每组记录分别计算 p_ts, mse, pearson
def process_entries(entries, label):
    rows = []
    res_ts = []  # list of p_ts arrays
    for idx, ent in enumerate(entries):
        vec = ent['vec']
        p_ts = param_to_ts(vec, M, SMOOTH_WINDOW, SMOOTH_POLY, normalize=NORMALIZE)
        try:
            mse = float(np.mean((p_ts - music_ts)**2))
        except Exception:
            mse = float('nan')
        try:
            pr, pval = pearsonr(p_ts, music_ts)
        except Exception:
            pr, pval = float('nan'), float('nan')
        rec = {'idx': idx, 'line_idx': ent['line_idx'], 'sub': ent['sub'],
               'fold': ent['fold'], 'epoch': ent['epoch'] if ent['epoch'] is not None else idx,
               'mse': mse, 'pearson': pr}
        rows.append(rec)
        res_ts.append(p_ts)
        # save resampled
        out_subdir = os.path.join(OUT_DIR, f"{label}_record_{idx}_sub{ent['sub']}_fold{ent['fold']}")
        os.makedirs(out_subdir, exist_ok=True)
        np.savetxt(os.path.join(out_subdir, "param_resampled.txt"), p_ts, fmt="%.6f")
    return rows, res_ts

rows_main, ts_main = process_entries(entries_main, 'main')
rows_orig, ts_orig = process_entries(entries_orig, 'orig')

# 3) 保存 CSV summaries
def save_csv(rows, fname):
    csv_path = os.path.join(OUT_DIR, fname)
    with open(csv_path, 'w', newline='', encoding='utf-8') as cf:
        writer = csv.writer(cf)
        writer.writerow(['index','line_idx','sub','fold','epoch','mse','pearson'])
        for r in rows:
            writer.writerow([r['idx'], r['line_idx'], r['sub'], r['fold'], r['epoch'], f"{r['mse']:.6e}", f"{r['pearson']:.6f}"])
save_csv(rows_main, "results_summary_main.csv")
save_csv(rows_orig,  "results_summary_orig.csv")

# 4) 绘制趋势对比（MSE & Pearson）—— 两条曲线分别为 main 与 orig
plt.figure(figsize=(8,4))
if len(rows_main)>0:
    indices_main = [r['epoch'] for r in rows_main]
    mse_main = [r['mse'] for r in rows_main]
    plt.plot(indices_main, mse_main, marker='o', linestyle='-', label='MSE (main params)')
if len(rows_orig)>0:
    indices_orig = [r['epoch'] for r in rows_orig]
    mse_orig = [r['mse'] for r in rows_orig]
    plt.plot(indices_orig, mse_orig, marker='s', linestyle='--', label='MSE (orig loss params)')
plt.xlabel("epoch / record index")
plt.ylabel("MSE (param vs music)")
plt.title("Param vs Music — MSE trend (lower is better)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "trend_mse_comparison.png"), dpi=300)
plt.close()

plt.figure(figsize=(8,4))
if len(rows_main)>0:
    pear_main = [r['pearson'] for r in rows_main]
    plt.plot(indices_main, pear_main, marker='o', linestyle='-', label='Pearson (main params)')
if len(rows_orig)>0:
    pear_orig = [r['pearson'] for r in rows_orig]
    plt.plot(indices_orig, pear_orig, marker='s', linestyle='--', label='Pearson (orig loss params)')
plt.xlabel("epoch / record index")
plt.ylabel("Pearson correlation (param vs music)")
plt.title("Param vs Music — Pearson trend (higher is better)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "trend_pearson_comparison.png"), dpi=300)
plt.close()

# 5) overlay 图（first / mid / last）—— 每张图画三条：main / orig / music
def get_three_ts(idx_main):
    """给定 main 的记录索引，找到对应的 main_ts, 最接近 epoch 的 orig_ts（若有），以及 music_ts"""
    p_main = ts_main[idx_main] if (0 <= idx_main < len(ts_main)) else None
    # find matching orig by epoch if possible
    p_orig = None
    if len(rows_main)>0 and len(rows_orig)>0:
        epoch_target = rows_main[idx_main]['epoch']
        # find orig with same epoch, else closest by absolute epoch diff
        orig_epochs = [r['epoch'] for r in rows_orig]
        diffs = [abs((e - epoch_target)) for e in orig_epochs]
        best_idx = int(np.argmin(diffs))
        p_orig = ts_orig[best_idx]
    return p_main, p_orig, music_ts

def plot_overlay_three(p_main, p_orig, music_ts, out_png, title):
    T = len(music_ts)
    t = np.linspace(0, T-1, T)
    plt.figure(figsize=(12,3))
    if p_main is not None:
        plt.plot(t, p_main, label='Main params', linewidth=0.9)
    if p_orig is not None:
        plt.plot(t, p_orig, label='Orig-loss params', linewidth=0.9, linestyle='--')
    plt.plot(t, music_ts, label='Music RMS', linewidth=0.9, alpha=0.9)
    plt.legend()
    plt.xlabel("frame")
    plt.ylabel("z-score" if NORMALIZE=='z' else "amplitude")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

# choose indices: first / mid / last of main if available; if main empty use orig indices
if len(ts_main) > 0:
    idxs = [0, len(ts_main)//2, len(ts_main)-1]
elif len(ts_orig) > 0:
    idxs = [0, len(ts_orig)//2, len(ts_orig)-1]
else:
    idxs = []

names = ['first', 'middle', 'last']
for name, idx in zip(names, idxs):
    p_main, p_orig, mus = get_three_ts(idx)
    outpng = os.path.join(OUT_DIR, f"overlay_{name}.png")
    title = f"Overlay — {name} record (main idx={idx})"
    plot_overlay_three(p_main, p_orig, mus, outpng, title)

print("输出保存在：", OUT_DIR)
print("已保存文件示例： trend_mse_comparison.png, trend_pearson_comparison.png, overlay_first.png / overlay_middle.png / overlay_last.png")


音乐帧数 M=35889. main entries: 2, orig entries: 2
输出保存在： C:\Users\33037\Desktop\论文\论文数据\模型参数的波形图
已保存文件示例： trend_mse_comparison.png, trend_pearson_comparison.png, overlay_first.png / overlay_middle.png / overlay_last.png


In [9]:
# plot_param_training_trend_with_orig.py
"""
同时读取 param_log.txt 与 orig_loss_param_log.txt，绘制三条曲线（主参数 / 原始损失生成参数 / 音乐RMS）。
"""
import os, re, csv
import numpy as np
from scipy.signal import savgol_filter
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# ========== 配置（请按需修改路径） ==========
PARAM_LOG_MAIN = r"C:\Users\33037\Desktop\画图与波形图三个曲线版\save\param_log.txt"
PARAM_LOG_ORIG = r"C:\Users\33037\Desktop\画图与波形图三个曲线版\save\orig_loss_param_log.txt"
MUSIC_RMS_PATH = r"C:\Users\33037\Desktop\论文\论文数据\判断音乐\RMS文件\music7_envelope.npy"  # .npy 或 .npz
OUT_DIR = r"C:\Users\33037\Desktop\论文\论文数据\模型参数的波形图"
SMOOTH_WINDOW = 11   # savgol window length (odd)
SMOOTH_POLY = 3
NORMALIZE = 'z'      # 'z' 或 None
DEFAULT_M = 2048     # 当无法读取音乐时的重采样长度
# ==========================================

os.makedirs(OUT_DIR, exist_ok=True)

def load_music_rms(path):
    if path is None or not os.path.exists(path):
        return None
    if path.endswith('.npy'):
        arr = np.load(path)
        return np.array(arr, dtype=float)
    elif path.endswith('.npz'):
        d = np.load(path)
        # 优先取 rms_z, 再 rms
        if 'rms_z' in d: return np.array(d['rms_z'], dtype=float)
        if 'rms' in d: return np.array(d['rms'], dtype=float)
        # 否则取第一个数组
        key = list(d.keys())[0]
        return np.array(d[key], dtype=float)
    else:
        # 非 np 文件，不尝试读取
        return None

def zscore(x):
    x = np.array(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)

def parse_param_log(path):
    """
    解析日志文件中每行的 SUB/FOLD/PARAMS，返回 entries 列表:
      [{'line_idx':i,'sub':sub,'fold':fold,'epoch':epoch,'vec':np.array}, ...]
    若找不到 SUB/FOLD，会把 sub/fold 设为 None。
    """
    if path is None or not os.path.exists(path):
        return []
    pat = re.compile(r"SUB\s*[:=]\s*(\d+)\s*FOLD\s*[:=]\s*(\d+).*?PARAMS\s*[:=]\s*(.+)", flags=re.IGNORECASE)
    ep_pat = re.compile(r"(?:EP|Epoch|EPOCH|epoch|step)\s*[:=]?\s*(\d+)", flags=re.IGNORECASE)
    entries = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            m = pat.search(line)
            if not m:
                # 尝试只找 PARAMS:
                m2 = re.search(r"PARAMS\s*[:=]\s*(.+)", line, flags=re.IGNORECASE)
                if not m2:
                    continue
                sub = None; fold = None
                params_str = m2.group(1).strip()
            else:
                sub = int(m.group(1)); fold = int(m.group(2))
                params_str = m.group(3).strip()
            ep_m = ep_pat.search(line)
            epoch = int(ep_m.group(1)) if ep_m else None
            toks = [t.strip() for t in params_str.split(',') if t.strip()!='']
            vals = []
            for t in toks:
                try:
                    vals.append(float(t))
                except:
                    pass
            if len(vals) == 0:
                continue
            entries.append({'line_idx': i, 'sub': sub, 'fold': fold, 'epoch': epoch, 'vec': np.array(vals, dtype=float)})
    return entries

def param_to_ts(vec, M, smooth_window=11, poly=3, normalize='z'):
    v = np.array(vec, dtype=float)
    if normalize == 'z':
        v = zscore(v)
    # savgol 平滑（若足够长）
    w = smooth_window if (smooth_window % 2 == 1) else smooth_window+1
    if len(v) >= w and w >= 3:
        try:
            v = savgol_filter(v, window_length=w, polyorder=poly)
        except Exception:
            pass
    # 插值到长度 M
    t_src = np.linspace(0,1,num=len(v))
    t_tgt = np.linspace(0,1,num=M)
    res = np.interp(t_tgt, t_src, v)
    if normalize == 'z':
        res = zscore(res)
    return res

# ---------- 读取数据 ----------
entries_main = parse_param_log(PARAM_LOG_MAIN)
entries_orig = parse_param_log(PARAM_LOG_ORIG)

if len(entries_main) == 0:
    raise RuntimeError("未在 param_log.txt 中解析到任何条目，请检查 PATH。")

music_rms = load_music_rms(MUSIC_RMS_PATH)
if music_rms is None:
    print("WARN: 未能加载音乐 RMS，使用默认长度 DEFAULT_M =", DEFAULT_M)
    M = DEFAULT_M
else:
    M = len(music_rms)

if NORMALIZE == 'z' and music_rms is not None:
    music_ts = zscore(music_rms)
elif music_rms is not None:
    music_ts = music_rms.copy()
else:
    music_ts = np.zeros(M, dtype=float)

print(f"Parsed main entries: {len(entries_main)}, orig entries: {len(entries_orig)}, music frames M={M}")

# ---------- 为两组 entries 生成重采样时序并计算指标 ----------
rows_main = []
for idx, ent in enumerate(entries_main):
    vec = ent['vec']
    p_ts = param_to_ts(vec, M, SMOOTH_WINDOW, SMOOTH_POLY, normalize=NORMALIZE)
    mse = float(np.mean((p_ts - music_ts)**2))
    try:
        pr, pval = pearsonr(p_ts, music_ts)
    except Exception:
        pr, pval = float('nan'), float('nan')
    rec = {'idx': idx, 'line_idx': ent['line_idx'], 'sub': ent['sub'], 'fold': ent['fold'],
           'epoch': ent['epoch'] if ent['epoch'] is not None else idx, 'mse': mse, 'pearson': pr, 'pearson_p': pval}
    rows_main.append((rec, p_ts))
    # save per-recorded resampled param
    out_subdir = os.path.join(OUT_DIR, f"main_record_{idx}_sub{ent['sub']}_fold{ent['fold']}")
    os.makedirs(out_subdir, exist_ok=True)
    np.savetxt(os.path.join(out_subdir, "param_resampled.txt"), p_ts, fmt="%.6f")

rows_orig = []
for idx, ent in enumerate(entries_orig):
    vec = ent['vec']
    p_ts = param_to_ts(vec, M, SMOOTH_WINDOW, SMOOTH_POLY, normalize=NORMALIZE)
    mse = float(np.mean((p_ts - music_ts)**2))
    try:
        pr, pval = pearsonr(p_ts, music_ts)
    except Exception:
        pr, pval = float('nan'), float('nan')
    rec = {'idx': idx, 'line_idx': ent['line_idx'], 'sub': ent['sub'], 'fold': ent['fold'],
           'epoch': ent['epoch'] if ent['epoch'] is not None else idx, 'mse': mse, 'pearson': pr, 'pearson_p': pval}
    rows_orig.append((rec, p_ts))
    out_subdir = os.path.join(OUT_DIR, f"orig_record_{idx}_sub{ent['sub']}_fold{ent['fold']}")
    os.makedirs(out_subdir, exist_ok=True)
    np.savetxt(os.path.join(out_subdir, "orig_param_resampled.txt"), p_ts, fmt="%.6f")

# ---------- 保存汇总 CSV ----------
csv_path = os.path.join(OUT_DIR, "results_summary_main.csv")
with open(csv_path, 'w', newline='', encoding='utf-8') as cf:
    writer = csv.writer(cf)
    writer.writerow(['index','line_idx','sub','fold','epoch','mse','pearson','pearson_p'])
    for rec, _ in rows_main:
        writer.writerow([rec['idx'], rec['line_idx'], rec['sub'], rec['fold'], rec['epoch'],
                         f"{rec['mse']:.6e}", f"{rec['pearson']:.6f}", rec['pearson_p']])

csv_path2 = os.path.join(OUT_DIR, "results_summary_orig.csv")
with open(csv_path2, 'w', newline='', encoding='utf-8') as cf:
    writer = csv.writer(cf)
    writer.writerow(['index','line_idx','sub','fold','epoch','mse','pearson','pearson_p'])
    for rec, _ in rows_orig:
        writer.writerow([rec['idx'], rec['line_idx'], rec['sub'], rec['fold'], rec['epoch'],
                         f"{rec['mse']:.6e}", f"{rec['pearson']:.6f}", rec['pearson_p']])

# ---------- 绘制趋势（MSE / Pearson） 同时显示 main 与 orig 两条曲线 ----------
indices_main = [r[0]['epoch'] for r in rows_main]
mse_main = [r[0]['mse'] for r in rows_main]
pear_main = [r[0]['pearson'] for r in rows_main]

plt.figure(figsize=(8,4))
plt.plot(indices_main, mse_main, marker='o', linestyle='-', label='Main MSE')
if len(rows_orig) > 0:
    indices_orig = [r[0]['epoch'] for r in rows_orig]
    mse_orig = [r[0]['mse'] for r in rows_orig]
    plt.plot(indices_orig, mse_orig, marker='x', linestyle='--', label='Orig-loss MSE')
plt.xlabel("epoch / record index")
plt.ylabel("MSE (param vs music)")
plt.title("MSE trend (lower is better)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "trend_mse.png"), dpi=300)
plt.close()

plt.figure(figsize=(8,4))
plt.plot(indices_main, pear_main, marker='o', linestyle='-', label='Main Pearson')
if len(rows_orig) > 0:
    pear_orig = [r[0]['pearson'] for r in rows_orig]
    plt.plot(indices_orig, pear_orig, marker='x', linestyle='--', label='Orig-loss Pearson')
plt.xlabel("epoch / record index")
plt.ylabel("Pearson (param vs music)")
plt.title("Pearson trend (higher is better)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "trend_pearson.png"), dpi=300)
plt.close()

# ---------- Overlay：绘制 first / mid / last 三图（每图包含三条曲线） ----------
def find_matching_orig(sub, fold):
    """按 sub/fold 找到 orig 的索引（优先匹配），没有返回 nearest index 或 None"""
    if len(rows_orig) == 0:
        return None
    for i, (rec, ts) in enumerate(rows_orig):
        if rec['sub'] == sub and rec['fold'] == fold:
            return i
    # 没有 exact match -> 返回靠近的 index（例如相同行号或最近的）
    # 简单策略：返回同索引号（若存在）
    return None  # 先不强行匹配，overlay 时若 None 使用全局中间项

def plot_overlay(main_ts, orig_ts, music_ts, out_png, title):
    T = len(main_ts)
    t = np.linspace(0, T-1, T)
    plt.figure(figsize=(10,2.6))
    plt.plot(t, main_ts, label='Main params', linewidth=1)
    if orig_ts is not None:
        plt.plot(t, orig_ts, linestyle='--', label='Orig-loss params', linewidth=1.5)
    plt.plot(t, music_ts, label='Music RMS', linewidth=1, alpha=0.9)
    plt.legend(loc='upper center')
    plt.xlabel("frame")
    plt.ylabel("z-score")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

# 选取 three indices 基于 main rows
first_idx = 0
mid_idx = len(rows_main)//2
last_idx = len(rows_main)-1
sel = [first_idx, mid_idx, last_idx]
names = ['first', 'middle', 'last']

for i_sel, name in zip(sel, names):
    rec_main, ts_main = rows_main[i_sel]
    # 尝试匹配 orig by sub/fold
    matched_i = None
    if rec_main['sub'] is not None and rec_main['fold'] is not None:
        for j, (rec_o, ts_o) in enumerate(rows_orig):
            if rec_o['sub'] == rec_main['sub'] and rec_o['fold'] == rec_main['fold']:
                matched_i = j
                break
    # 若未匹配到，尝试按同索引（若存在）
    if matched_i is None and i_sel < len(rows_orig):
        matched_i = i_sel
    orig_ts = rows_orig[matched_i][1] if (matched_i is not None and matched_i < len(rows_orig)) else None

    # 如果 music_ts 长度跟 ts_main 区别很大，已经在 param_to_ts 中把所有 ts 重采样到 M（music length）。
    plot_overlay(ts_main, orig_ts, music_ts, os.path.join(OUT_DIR, f"overlay_{name}.png"),
                 f"Overlay — {name} record (main idx={i_sel})")

print("输出保存在：", OUT_DIR)
print("说明：")
print(" - overlay_xxx.png: 三条线分别为 主参数、原始损失生成参数（若存在）和音乐 RMS；横轴为 frame（已重采样到音乐帧数）。")
print(" - trend_mse.png / trend_pearson.png: 同时展示 Main 与 Orig-loss 的趋势（若 orig 存在）。")
print(" - param_resampled.txt 与 orig_param_resampled.txt 已分别保存每条记录的重采样向量。")


Parsed main entries: 2, orig entries: 2, music frames M=35889
输出保存在： C:\Users\33037\Desktop\论文\论文数据\模型参数的波形图
说明：
 - overlay_xxx.png: 三条线分别为 主参数、原始损失生成参数（若存在）和音乐 RMS；横轴为 frame（已重采样到音乐帧数）。
 - trend_mse.png / trend_pearson.png: 同时展示 Main 与 Orig-loss 的趋势（若 orig 存在）。
 - param_resampled.txt 与 orig_param_resampled.txt 已分别保存每条记录的重采样向量。


In [ ]:
# plot_param_training_trend_with_orig.py
"""
同时读取 param_log.txt 与 orig_loss_param_log.txt，绘制三条曲线（主参数 / 原始损失生成参数 / 音乐RMS）。
"""
import os, re, csv
import numpy as np
from scipy.signal import savgol_filter
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# ========== 配置（请按需修改路径） ==========
PARAM_LOG_MAIN = r"C:\Users\33037\Desktop\画图与波形图三个曲线版\save\param_log.txt"
PARAM_LOG_ORIG = r"C:\Users\33037\Desktop\画图与波形图三个曲线版\save\orig_loss_param_log.txt"
MUSIC_RMS_PATH = r"C:\Users\33037\Desktop\论文\论文数据\判断音乐\RMS文件\music7_envelope.npy"  # .npy 或 .npz
OUT_DIR = r"C:\Users\33037\Desktop\论文\论文数据\模型参数的波形图"
SMOOTH_WINDOW = 11   # savgol window length (odd)
SMOOTH_POLY = 3
NORMALIZE = 'z'      # 'z' 或 None
DEFAULT_M = 2048     # 当无法读取音乐时的重采样长度

# ========== 新增配置：统一 X 轴显示区间 ==========
X_MIN = 7500
X_MAX = 12500  # 将所有图的 X 轴范围设为 X_MIN ~ X_MAX
# ==========================================

os.makedirs(OUT_DIR, exist_ok=True)

def load_music_rms(path):
    if path is None or not os.path.exists(path):
        return None
    if path.endswith('.npy'):
        arr = np.load(path)
        return np.array(arr, dtype=float)
    elif path.endswith('.npz'):
        d = np.load(path)
        # 优先取 rms_z, 再 rms
        if 'rms_z' in d: return np.array(d['rms_z'], dtype=float)
        if 'rms' in d: return np.array(d['rms'], dtype=float)
        # 否则取第一个数组
        key = list(d.keys())[0]
        return np.array(d[key], dtype=float)
    else:
        # 非 np 文件，不尝试读取
        return None

def zscore(x):
    x = np.array(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)

def parse_param_log(path):
    """
    解析日志文件中每行的 SUB/FOLD/PARAMS，返回 entries 列表:
      [{'line_idx':i,'sub':sub,'fold':fold,'epoch':epoch,'vec':np.array}, ...]
    若找不到 SUB/FOLD，会把 sub/fold 设为 None。
    """
    if path is None or not os.path.exists(path):
        return []
    pat = re.compile(r"SUB\s*[:=]\s*(\d+)\s*FOLD\s*[:=]\s*(\d+).*?PARAMS\s*[:=]\s*(.+)", flags=re.IGNORECASE)
    ep_pat = re.compile(r"(?:EP|Epoch|EPOCH|epoch|step)\s*[:=]?\s*(\d+)", flags=re.IGNORECASE)
    entries = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            m = pat.search(line)
            if not m:
                # 尝试只找 PARAMS:
                m2 = re.search(r"PARAMS\s*[:=]\s*(.+)", line, flags=re.IGNORECASE)
                if not m2:
                    continue
                sub = None; fold = None
                params_str = m2.group(1).strip()
            else:
                sub = int(m.group(1)); fold = int(m.group(2))
                params_str = m.group(3).strip()
            ep_m = ep_pat.search(line)
            epoch = int(ep_m.group(1)) if ep_m else None
            toks = [t.strip() for t in params_str.split(',') if t.strip()!='']
            vals = []
            for t in toks:
                try:
                    vals.append(float(t))
                except:
                    pass
            if len(vals) == 0:
                continue
            entries.append({'line_idx': i, 'sub': sub, 'fold': fold, 'epoch': epoch, 'vec': np.array(vals, dtype=float)})
    return entries

def param_to_ts(vec, M, smooth_window=11, poly=3, normalize='z'):
    v = np.array(vec, dtype=float)
    if normalize == 'z':
        v = zscore(v)
    # savgol 平滑（若足够长）
    w = smooth_window if (smooth_window % 2 == 1) else smooth_window+1
    if len(v) >= w and w >= 3:
        try:
            v = savgol_filter(v, window_length=w, polyorder=poly)
        except Exception:
            pass
    # 插值到长度 M
    t_src = np.linspace(0,1,num=len(v))
    t_tgt = np.linspace(0,1,num=M)
    res = np.interp(t_tgt, t_src, v)
    if normalize == 'z':
        res = zscore(res)
    return res

# ---------- 读取数据 ----------
entries_main = parse_param_log(PARAM_LOG_MAIN)
entries_orig = parse_param_log(PARAM_LOG_ORIG)

if len(entries_main) == 0:
    raise RuntimeError("未在 param_log.txt 中解析到任何条目，请检查 PATH。")

music_rms = load_music_rms(MUSIC_RMS_PATH)
if music_rms is None:
    print("WARN: 未能加载音乐 RMS，使用默认长度 DEFAULT_M =", DEFAULT_M)
    M = DEFAULT_M
else:
    M = len(music_rms)

if NORMALIZE == 'z' and music_rms is not None:
    music_ts = zscore(music_rms)
elif music_rms is not None:
    music_ts = music_rms.copy()
else:
    music_ts = np.zeros(M, dtype=float)

print(f"Parsed main entries: {len(entries_main)}, orig entries: {len(entries_orig)}, music frames M={M}")

# ---------- 为两组 entries 生成重采样时序并计算指标 ----------
rows_main = []
for idx, ent in enumerate(entries_main):
    vec = ent['vec']
    p_ts = param_to_ts(vec, M, SMOOTH_WINDOW, SMOOTH_POLY, normalize=NORMALIZE)
    mse = float(np.mean((p_ts - music_ts)**2))
    try:
        pr, pval = pearsonr(p_ts, music_ts)
    except Exception:
        pr, pval = float('nan'), float('nan')
    rec = {'idx': idx, 'line_idx': ent['line_idx'], 'sub': ent['sub'], 'fold': ent['fold'],
           'epoch': ent['epoch'] if ent['epoch'] is not None else idx, 'mse': mse, 'pearson': pr, 'pearson_p': pval}
    rows_main.append((rec, p_ts))
    # save per-recorded resampled param
    out_subdir = os.path.join(OUT_DIR, f"main_record_{idx}_sub{ent['sub']}_fold{ent['fold']}")
    os.makedirs(out_subdir, exist_ok=True)
    np.savetxt(os.path.join(out_subdir, "param_resampled.txt"), p_ts, fmt="%.6f")

rows_orig = []
for idx, ent in enumerate(entries_orig):
    vec = ent['vec']
    p_ts = param_to_ts(vec, M, SMOOTH_WINDOW, SMOOTH_POLY, normalize=NORMALIZE)
    mse = float(np.mean((p_ts - music_ts)**2))
    try:
        pr, pval = pearsonr(p_ts, music_ts)
    except Exception:
        pr, pval = float('nan'), float('nan')
    rec = {'idx': idx, 'line_idx': ent['line_idx'], 'sub': ent['sub'], 'fold': ent['fold'],
           'epoch': ent['epoch'] if ent['epoch'] is not None else idx, 'mse': mse, 'pearson': pr, 'pearson_p': pval}
    rows_orig.append((rec, p_ts))
    out_subdir = os.path.join(OUT_DIR, f"orig_record_{idx}_sub{ent['sub']}_fold{ent['fold']}")
    os.makedirs(out_subdir, exist_ok=True)
    np.savetxt(os.path.join(out_subdir, "orig_param_resampled.txt"), p_ts, fmt="%.6f")

# ---------- 保存汇总 CSV ----------
csv_path = os.path.join(OUT_DIR, "results_summary_main.csv")
with open(csv_path, 'w', newline='', encoding='utf-8') as cf:
    writer = csv.writer(cf)
    writer.writerow(['index','line_idx','sub','fold','epoch','mse','pearson','pearson_p'])
    for rec, _ in rows_main:
        writer.writerow([rec['idx'], rec['line_idx'], rec['sub'], rec['fold'], rec['epoch'],
                         f"{rec['mse']:.6e}", f"{rec['pearson']:.6f}", rec['pearson_p']])

csv_path2 = os.path.join(OUT_DIR, "results_summary_orig.csv")
with open(csv_path2, 'w', newline='', encoding='utf-8') as cf:
    writer = csv.writer(cf)
    writer.writerow(['index','line_idx','sub','fold','epoch','mse','pearson','pearson_p'])
    for rec, _ in rows_orig:
        writer.writerow([rec['idx'], rec['line_idx'], rec['sub'], rec['fold'], rec['epoch'],
                         f"{rec['mse']:.6e}", f"{rec['pearson']:.6f}", rec['pearson_p']])

# ---------- 绘制趋势（MSE / Pearson） 同时显示 main 与 orig 两条曲线 ----------
indices_main = [r[0]['epoch'] for r in rows_main]
mse_main = [r[0]['mse'] for r in rows_main]
pear_main = [r[0]['pearson'] for r in rows_main]

plt.figure(figsize=(8,4))
plt.plot(indices_main, mse_main, marker='o', linestyle='-', label='Main MSE')
if len(rows_orig) > 0:
    indices_orig = [r[0]['epoch'] for r in rows_orig]
    mse_orig = [r[0]['mse'] for r in rows_orig]
    plt.plot(indices_orig, mse_orig, marker='x', linestyle='--', label='Orig-loss MSE')
plt.xlabel("epoch / record index")
plt.ylabel("MSE (param vs music)")
plt.title("MSE trend (lower is better)")
# 将图例放到图片下方居中，并以三列显示
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3)
# 设置 x 轴范围为 X_MIN ~ X_MAX
plt.xlim(X_MIN, X_MAX)
plt.grid(True)
plt.tight_layout()
# 调整底部空白以容纳图例，保存时确保包含图例
plt.subplots_adjust(bottom=0.25)
plt.savefig(os.path.join(OUT_DIR, "trend_mse.png"), dpi=300, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8,4))
plt.plot(indices_main, pear_main, marker='o', linestyle='-', label='Main Pearson')
if len(rows_orig) > 0:
    pear_orig = [r[0]['pearson'] for r in rows_orig]
    plt.plot(indices_orig, pear_orig, marker='x', linestyle='--', label='Orig-loss Pearson')
plt.xlabel("epoch / record index")
plt.ylabel("Pearson (param vs music)")
plt.title("Pearson trend (higher is better)")
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3)
plt.xlim(X_MIN, X_MAX)
plt.grid(True)
plt.tight_layout()
plt.subplots_adjust(bottom=0.25)
plt.savefig(os.path.join(OUT_DIR, "trend_pearson.png"), dpi=300, bbox_inches='tight')
plt.close()

# ---------- Overlay：绘制 first / mid / last 三图（每图包含三条曲线） ----------
def find_matching_orig(sub, fold):
    """按 sub/fold 找到 orig 的索引（优先匹配），没有返回 nearest index 或 None"""
    if len(rows_orig) == 0:
        return None
    for i, (rec, ts) in enumerate(rows_orig):
        if rec['sub'] == sub and rec['fold'] == fold:
            return i
    # 没有 exact match -> 返回靠近的 index（例如相同行号或最近的）
    # 简单策略：返回同索引号（若存在）
    return None  # 先不强行匹配，overlay 时若 None 使用全局中间项

def plot_overlay(main_ts, orig_ts, music_ts, out_png, title):
    T = len(main_ts)
    t = np.linspace(0, T-1, T)
    plt.figure(figsize=(10,2.6))
    plt.plot(t, main_ts, label='Main params', linewidth=1)
    if orig_ts is not None:
        plt.plot(t, orig_ts, linestyle='--', label='Orig-loss params', linewidth=1.5)
    plt.plot(t, music_ts, label='Music RMS', linewidth=1, alpha=0.9)
    # 把图例放到图下方居中，三列显示
    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=3)
    plt.xlabel("frame")
    plt.ylabel("z-score")
    plt.title(title)
    # 强制 x 轴范围 X_MIN ~ X_MAX
    plt.xlim(X_MIN, X_MAX)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.28)
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close()

# 选取 three indices 基于 main rows
first_idx = 0
mid_idx = len(rows_main)//2
last_idx = len(rows_main)-1
sel = [first_idx, mid_idx, last_idx]
names = ['first', 'middle', 'last']

for i_sel, name in zip(sel, names):
    rec_main, ts_main = rows_main[i_sel]
    # 尝试匹配 orig by sub/fold
    matched_i = None
    if rec_main['sub'] is not None and rec_main['fold'] is not None:
        for j, (rec_o, ts_o) in enumerate(rows_orig):
            if rec_o['sub'] == rec_main['sub'] and rec_o['fold'] == rec_main['fold']:
                matched_i = j
                break
    # 若未匹配到，尝试按同索引（若存在）
    if matched_i is None and i_sel < len(rows_orig):
        matched_i = i_sel
    orig_ts = rows_orig[matched_i][1] if (matched_i is not None and matched_i < len(rows_orig)) else None

    # 如果 music_ts 长度跟 ts_main 区别很大，已经在 param_to_ts 中把所有 ts 重采样到 M（music length）。
    plot_overlay(ts_main, orig_ts, music_ts, os.path.join(OUT_DIR, f"overlay_{name}.png"),
                 f"Overlay — {name} record (main idx={i_sel})")

print("输出保存在：", OUT_DIR)
print("说明：")
print(" - overlay_xxx.png: 三条线分别为 主参数、原始损失生成参数（若存在）和音乐 RMS；横轴为 frame（已重采样到音乐帧数）。")
print(" - trend_mse.png / trend_pearson.png: 同时展示 Main 与 Orig-loss 的趋势（若 orig 存在）。")
print(" - param_resampled.txt 与 orig_param_resampled.txt 已分别保存每条记录的重采样向量。")
